# Modeling Power Outages by County in Maine Using Manual Lags

The goal of this notebook is to predict customers.out in the counties in Maine by manually adding lagged data to the dataset.

Maine has only sixteen counties with a wide range of topography and population. 
If we are able to set up hierarchical methods for ME, we should be able to scale up to the entire US.

The outline of the notebook is:
- Setup
    - Import packages
    - Load and arrange the data


## Imports

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

from sklearn.model_selection import KFold

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error


from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

## Load and Arrange the Data

We'll load the dataset with eaglei, noaa, era5, county-level, and engineered variables

We'll restrict to our training set (pre-2022)

For now, we'll also restrict ourselves to Maine, which has just 16 counties--a sufficient number to let us try out the techniques before deploying on the entire country

In [ ]:
#Load the engineered data
df = pd.read_parquet('../Data/Merged_Data/eaglei_noaa_era5_engineered.parquet')

# Sort df
df = df.sort_values(by=['fips_code', 'datetime'])

# Restrict df to years prior to 2022
# For now, we'll just focus on Maine
df = df[df['STUSPS'] == 'ME']
df = df[df['YEAR'] < 2022]

# Restrict to a smaller data set for testing purposes
df_small = df.groupby(level=0).tail(200)

# The first nonzero entry for customers_out in Maine was November 2nd, 2014
# We won't need to call this using the small df
#df = df[df['datetime'] > '2014-11-01']

# Use fips_code and datetime as indices
df = df.set_index(['fips_code', 'datetime'])

df_small = df_small.set_index(['fips_code', 'datetime'])

# Note that datetime doesn't have an embedded frequency; we'll need to set this manually
# Not sure if we'll need this
#df.index = df.index.set_levels(df.index.levels[1].to_period('6h'), level=1)

# Adding Lags

In [4]:
lag_features = ['event_count SnowIce',
       'event_count Flood', 'event_count Storm', 'event_count Hurricane',
       'event_count Heat', 'event_count Fire', 'event_count Wind',
       'event_count Ocean', 'event_count Other', 't2m', 'sf', 'tp', 't2m_mean',
       't2m_max', 'sf_mean', 'sf_max', 'tp_mean', 'tp_max', 'wind_speed',
       'neighbor_mean_wind_speed', 'neighbor_max_wind_speed', 'sf_12h',
       'sf_24h', 'tp_12h', 'tp_24h']

# For each variable in lag_features, use the shift function to create three lagged values as a new variable
for feature in lag_features:
    df[feature + '_1'] = df[feature].shift(1)
    df[feature + '_2'] = df[feature].shift(2)
    df[feature + '_3'] = df[feature].shift(3)

### Generate a list of features

In [6]:
# Create a list of these newly added variables
lagged_features = []
for feature in lag_features:
    lagged_features.append(feature + '_1')
    lagged_features.append(feature + '_2')
    lagged_features.append(feature + '_3')

# Not including Subregion for now, since we'd need to convert it to a dummy variable

features = ['Pct_Buried_Lines', 'centroid_longitude', 'centroid_latitude',
            'POPULATION', 'AREA', 'event_count SnowIce', 'event_count Flood',
            'event_count Storm', 'event_count Hurricane', 'event_count Heat',
            'event_count Fire', 'event_count Wind', 'event_count Ocean',
            'event_count Other'] + lag_features + lagged_features


### Drop all rows with NaNs

The first couple of rows are missing lagged values, so will have NaNs

In [7]:
df = df.dropna(subset=features)

### Create a smaller version of df for testing

In [14]:
# Restrict to a smaller data set for testing purposes
df_small = df.groupby(level=0).tail(200)

# Note that datetime doesn't have an embedded frequency; we'll need to set this manually
# Not sure if we'll need this
#df.index = df.index.set_levels(df.index.levels[1].to_period('6h'), level=1)

## Random Forest for Feature Importance

Basically nothing is particularly important, although it looks like wind is a better predictor than other things...

In [9]:
rf = RandomForestRegressor(n_estimators = 300, max_features = 'sqrt', max_depth = 5, random_state = 216)

rf.fit(df[features],df['customers_out'])

pred = rf.predict(df[features])

#Get feature importance from the rf
score = pd.DataFrame({'feature':df[features].columns,
                            'importance_score': rf.feature_importances_})

score.sort_values('importance_score',ascending=False)

,feature,importance_score
58,event_count Wind_2,0.036623
59,event_count Wind_3,0.032141
57,event_count Wind_1,0.031807
3,POPULATION,0.031394
96,neighbor_mean_wind_speed_1,0.026399
...,...,...
17,event_count Hurricane,0.000000
51,event_count Heat_1,0.000000
52,event_count Heat_2,0.000000
18,event_count Heat,0.000000


# Model Comparison

In [ ]:
# Copy the fips_code values as a new variable; we'll use this for stratification
df['fips'] = df.index.get_level_values(0)
df_small['fips'] = df_small.index.get_level_values(0)

### Set up Models

In [21]:
# List the various models we'll try to identify their "out of the box" performance
models = {
'mlr_pipe' : Pipeline([
                ('scale', StandardScaler()),
                ('linear', LinearRegression())]),

'knn_pipe' : Pipeline([
                ('scale', StandardScaler()),
                ('knn', KNeighborsRegressor(10))]),

'svr' : SVR(),

'rf': RandomForestRegressor(),

'ada' : AdaBoostRegressor(),

'grad' : GradientBoostingRegressor(),

# For some reason XGB is leading to an error with the fit command:
# 'DataFrame' object has no attribute 'dtype'
# Not sure exactly what's going on here; will figure it out later...
#'xgb': XGBRegressor(),

}

### Run each model on the lagged data

In [ ]:
# We'll stratify based on fips code

rmses = []

mlr = Pipeline([
                ('scale', StandardScaler()),
                ('linear', LinearRegression())])

for train_index, test_index in kfold.split(df[features], df['fips']):
    train_tt = df.iloc[train_index]
    train_ho = df.iloc[test_index]

    mlr.fit(df[features], df['customers_out'])
    pred = mlr.predict(train_ho[features])

    rmses.append(root_mean_squared_error(train_ho['customers_out'], pred))

In [25]:
# Set up a list of the models and methods to organize the computation of means in the kfold split
modellist = []
for pipeline_name, pipeline_obj in models.items():
    modellist.append(pipeline_name)

num_splits = 5

# Create an array with len(modellist) rows and len(methodlist) columns
output = np.zeros((len(modellist), num_splits))

#Make a StratifiedKFold object stratified by the variable sii
# This is necessary due to the small number of sii=3 values
kfold = StratifiedKFold(n_splits=num_splits, shuffle=True)

## i will count the split number 
i = 0
for train_index, test_index in kfold.split(df_small[features], df_small['fips']):
    train_tt = df_small.iloc[train_index]
    train_ho = df_small.iloc[test_index]

    # j will enumerate the model
    j=0

    for pipeline_name, pipeline_obj in models.items():
        # Fit and make predictions
        pipeline_obj.fit(df_small[features], df_small['customers_out'])
        pred = pipeline_obj.predict(train_ho[features])

        # Compute rmse
        rmse = root_mean_squared_error(train_ho['customers_out'], pred)
        
        # Store the kappa values in the output array
        output[j,i] = rmse
        j=j+1
    i=i+1

# Create a new array by computing the average of the values in output along the third axis
output_avg = np.mean(output, axis=1)

## Record the mean RMSE for each model

It looks like the gradient boosting regressor is performing well... for this small dataset

In [26]:
# create a data frame from output using modellist as the names of the columns and methodlist as the names of the rows
output_df = pd.DataFrame(output_avg, index=modellist)

output_df

,0
mlr_pipe,170.735646
knn_pipe,150.931305
svr,182.650121
rf,61.644617
ada,211.457458
grad,59.770226
